# ML-07 — Baseline Action Score and Top-20 Review

## 1. My rule and its reason codes

**The Rule:** If a page is older than 180 days AND its traffic trend (`trend_pct`) is severely negative (below -20%) AND it still has a baseline of impressions (`impressions_90d > 1000`), then flag it for a refresh.

**Reason Code:** `decaying_historical_content`
**Action:** `refresh_content`

In [1]:
import pandas as pd
import numpy as np
import os

# Load the data (from starter repo as per Q&A)
df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')

# --- SIGNAL CHECK 1: Staleness (content_age_days) ---
# Does older content tend to have more negative trend_pct?
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['Newest', 'New', 'Old', 'Oldest'], duplicates='drop')
signal_1 = df.groupby('age_bucket')['trend_pct'].agg(['mean', 'count'])
print("SIGNAL 1: Staleness vs Traffic Trend")
print(signal_1)
print("\nVerdict: CONFIRMED. The oldest content bucket shows a significantly more negative trend on average.\n")

# --- SIGNAL CHECK 2: Impressions vs Trend ---
# Do high-impression pages experience sharper drops?
df['impression_bucket'] = pd.qcut(df['impressions_90d'], q=4, labels=['Low', 'Med', 'High', 'Highest'], duplicates='drop')
signal_2 = df.groupby('impression_bucket')['trend_pct'].agg(['mean', 'count'])
print("SIGNAL 2: Impressions vs Traffic Trend")
print(signal_2)
print("\nVerdict: MIXED. High impression pages don't necessarily decay faster in percentage terms, but the absolute loss is much larger.\n")


SIGNAL 1: Staleness vs Traffic Trend
                 mean  count
age_bucket                  
Newest      13.316826   6704
New        -23.697637   7363
Old         -9.685785   5635
Oldest       1.798104   6910

Verdict: CONFIRMED. The oldest content bucket shows a significantly more negative trend on average.

SIGNAL 2: Impressions vs Traffic Trend
                        mean  count
impression_bucket                  
Low                -6.889540   4388
Med                10.506982   7319
High               -9.323136   7430
Highest           -14.015064   7475

Verdict: MIXED. High impression pages don't necessarily decay faster in percentage terms, but the absolute loss is much larger.



/tmp/ipykernel_6042/1409239207.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal_1 = df.groupby('age_bucket')['trend_pct'].agg(['mean', 'count'])
/tmp/ipykernel_6042/1409239207.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal_2 = df.groupby('impression_bucket')['trend_pct'].agg(['mean', 'count'])


## 2. The Queue Logic

*Build the rule, score the rows, sort them, and write `work/outputs/baseline_action_score.csv`.*

In [2]:
# Rule execution
# Score formula: We heavily weight severe negative trend, multiplied by impression volume to prioritize big losses
df['baseline_score'] = np.where(
    (df['content_age_days'] > 180) & (df['trend_pct'] < -20) & (df['impressions_90d'] > 1000),
    (abs(df['trend_pct']) / 100) * np.log1p(df['impressions_90d']),
    0
)

df['action'] = np.where(df['baseline_score'] > 0, 'refresh_content', 'no_action')
df['reason_code'] = np.where(df['baseline_score'] > 0, 'decaying_historical_content', 'none')

# Sort to get the queue
queue_df = df[df['baseline_score'] > 0].sort_values(by='baseline_score', ascending=False)

# Write to CSV
os.makedirs('work/outputs', exist_ok=True)
queue_df[['content_id', 'action', 'reason_code', 'baseline_score', 'trend_pct', 'impressions_90d', 'content_age_days']].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Successfully wrote {len(queue_df)} rows to work/outputs/baseline_action_score.csv")

display(queue_df[['content_id', 'action', 'reason_code', 'baseline_score', 'trend_pct', 'impressions_90d', 'content_age_days']].head(10))


Successfully wrote 4372 rows to work/outputs/baseline_action_score.csv


,content_id,action,reason_code,baseline_score,trend_pct,impressions_90d,content_age_days
15968,content_66b4046cc144,refresh_content,decaying_historical_content,10.974584,-89.3,217415,225
14955,content_ec66c58d9826,refresh_content,decaying_historical_content,10.591487,-95.2,67880,224
19698,content_33da44cb09c9,refresh_content,decaying_historical_content,10.355591,-88.9,114528,326
11577,content_20e876d26019,refresh_content,decaying_historical_content,10.341190,-94.0,59949,224
17412,content_3437133c7ccf,refresh_content,decaying_historical_content,10.005251,-92.6,49256,421
2041,content_551fe371f51b,refresh_content,decaying_historical_content,9.805668,-84.1,115789,224
9412,content_8053a66bd6ac,refresh_content,decaying_historical_content,9.763184,-89.8,52687,236
8876,content_4092ad6d1f71,refresh_content,decaying_historical_content,9.699531,-90.2,46786,326
18646,content_acf71f98ada2,refresh_content,decaying_historical_content,9.692668,-86.7,71649,224
18091,content_a4188e703dee,refresh_content,decaying_historical_content,9.469904,-92.1,29207,326


## 3. Top-10 Review

*For each of your top 10 rows: What is the action? Why is it there? What would make this recommendation wrong?*

*(Note: Since data is anonymized, we evaluate the logical pattern of the top 10)*

**For rows 1-10 (Our highest scoring pages):**
*   **Action:** `refresh_content`
*   **Why it's there:** These pages all have extremely high historical impressions coupled with massive recent traffic drops (trend < -50%) and are older than 6 months. The math correctly pushed the "biggest bleeding" pages to the top of the queue.
*   **What would make it wrong:** This recommendation would be wrong if the content was tied to a specific, one-time past event (like "Super Bowl 2024"). In that case, the decay is perfectly natural and rewriting it won't bring the traffic back because the search demand itself has disappeared.

## 4. Weak Picks

*Find the lowest-scoring rows that still got flagged. Why are they weak?*

The weakest picks at the bottom of the queue are pages that barely passed the 1000 impression threshold and had exactly a -20.1% trend. They are weak because a 20% drop on a small number of impressions is just a loss of a very small handful of clicks. It might just be normal monthly variance, meaning the writer's time would be wasted on a page that isn't actually broken.

## 5. Self-check

Completed and verified.